# Angular and kinematic studies

This notebook studies polarized W-pair production and decay, lepton kinematics, and the comparison between signal components and the full dilepton background. It also writes the ROOT templates used by the conventional likelihood notebook.

## Before running

1. Install and activate a Python environment in which `import ROOT` works.
2. Place the required ROOT input files in `data/`, or configure `FCC_INPUT_DIR`.
3. Adjust sample names, cross-sections, luminosity, and histogram ranges in `analysis_config.py` when needed.
4. Run all cells from top to bottom.

All generated ROOT files and plots are written to the configured output directory (default: `outputs/`).


## 1. ROOT setup, shared style, and imports

In [ ]:
"""Angular and kinematic studies for polarized W-pair samples.

The script produces:
  * the absolute W production-angle distribution;
  * charge-separated W+ and W- decay-angle distributions and stacked plots;
  * leading/subleading lepton pT and dilepton invariant-mass distributions;
  * signal-component versus full-background comparisons;
  * ROOT templates for the conventional binned likelihood analysis.

Input files, cross-sections, luminosity, tree name, and output locations are
configured in analysis_config.py. The script expects the input TTree to expose
the LHE particle branches used below.
"""

from collections import defaultdict
from pathlib import Path

import ROOT

from analysis_config import (
    COLLISION_ENERGY_GEV,
    CROSS_SECTIONS_PB,
    EVENTS_TREE_NAME,
    LUMI_PB,
    N_BINS,
    OBSERVABLE_RANGES,
    PLOTS_DIR,
    PRODUCTION_CROSS_SECTIONS_PB,
    TEMPLATE_FILE,
    sample_path,
)


# ---------------------------------------------------------------------------
# ROOT setup and common plot style
# ---------------------------------------------------------------------------
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptTitle(0)
ROOT.gStyle.SetPadTickX(1)
ROOT.gStyle.SetPadTickY(1)
ROOT.gStyle.SetEndErrorSize(0)
ROOT.gStyle.SetLabelFont(42, "xyz")
ROOT.gStyle.SetTitleFont(42, "xyz")
ROOT.gStyle.SetLabelSize(0.040, "xyz")
ROOT.gStyle.SetTitleSize(0.045, "xyz")
ROOT.gStyle.SetTitleOffset(1.4, "y")

COLORS = {
    "00": ROOT.TColor.GetColor("#0072B2"),
    "TT": ROOT.TColor.GetColor("#C0392B"),
    "mixed": ROOT.TColor.GetColor("#7B3294"),
    "bkg": ROOT.TColor.GetColor("#666666"),
    "total": ROOT.TColor.GetColor("#000000"),
}

LABELS = {
    "00": "Longitudinal (00)",
    "TT": "Transverse (TT)",
    "mixed": "Mixed (0T + T0)",
    "bkg": "Full background (ee, #mu#mu, e#mu)",
}

Y_TITLE = "Events / bin"
TITLE_COS_THETA_W = "|cos(#theta_{W})|"
TITLE_COS_THETA_STAR = "cos(#theta^{*})"
TITLE_LEADING_PT = "Leading lepton p_{T} [GeV]"
TITLE_SUBLEADING_PT = "Subleading lepton p_{T} [GeV]"
TITLE_MLL = "M_{ll} [GeV]"
FILL_ALPHA = 0.15

# Keep PyROOT objects referenced for as long as the script is running.
ROOT_OBJECTS = []
INPUT_FILES = []




## 2. Generic ROOT and plotting helpers

In [ ]:
# ---------------------------------------------------------------------------
# Generic ROOT helpers
# ---------------------------------------------------------------------------
def open_tree(role, tree_name=EVENTS_TREE_NAME):
    """Open a configured ROOT sample and return its file handle and TTree."""
    path = sample_path(role)
    root_file = ROOT.TFile.Open(path, "READ")
    if not root_file or root_file.IsZombie():
        raise OSError(f"Could not open ROOT input file: {path}")

    tree = root_file.Get(tree_name)
    if not tree:
        root_file.Close()
        raise KeyError(f"Tree {tree_name!r} not found in {path}")

    INPUT_FILES.append(root_file)
    return root_file, tree


def make_hist(name, x_title, x_min, x_max, y_title=Y_TITLE):
    """Create a weighted TH1F with the configured number of bins."""
    hist = ROOT.TH1F(name, f";{x_title};{y_title}", N_BINS, x_min, x_max)
    hist.Sumw2()
    hist.SetStats(False)
    ROOT_OBJECTS.append(hist)
    return hist


def make_canvas(name, title="", width=900, height=650):
    """Create a canvas with the common analysis margins."""
    canvas = ROOT.TCanvas(name, title, width, height)
    canvas.SetLeftMargin(0.15)
    canvas.SetBottomMargin(0.15)
    canvas.SetTopMargin(0.12)
    canvas.SetRightMargin(0.05)
    ROOT_OBJECTS.append(canvas)
    return canvas


def draw_experiment_label():
    """Draw the experiment label, collision energy, and integrated luminosity."""
    latex = ROOT.TLatex()
    latex.SetNDC()
    latex.SetTextFont(61)
    latex.SetTextSize(0.045)
    latex.DrawLatex(0.15, 0.92, "FCC-ee")
    latex.SetTextFont(52)
    latex.SetTextSize(0.035)
    latex.DrawLatex(0.28, 0.92, "Simulation")
    latex.SetTextFont(42)
    latex.SetTextSize(0.030)
    luminosity_ab = LUMI_PB / 1.0e6
    latex.DrawLatex(
        0.45,
        0.92,
        f"#sqrt{{s}} = {COLLISION_ENERGY_GEV:g} GeV, "
        f"L = {luminosity_ab:g} ab^{{-1}}",
    )
    ROOT_OBJECTS.append(latex)


def style_hist(hist, color, line_style=1, fill=True):
    """Apply the standard line and optional transparent-fill style."""
    hist.SetLineColor(color)
    hist.SetLineWidth(2)
    hist.SetLineStyle(line_style)
    hist.SetMarkerColor(color)
    if fill:
        hist.SetFillColorAlpha(color, FILL_ALPHA)


def make_legend(x1, y1, x2, y2, header=None, text_size=0.030):
    """Create a transparent legend without a border."""
    legend = ROOT.TLegend(x1, y1, x2, y2)
    legend.SetBorderSize(0)
    legend.SetFillStyle(0)
    legend.SetTextFont(42)
    legend.SetTextSize(text_size)
    if header:
        legend.SetHeader(header)
    ROOT_OBJECTS.append(legend)
    return legend


def scale_to_luminosity(hist, sigma_pb, n_generated):
    """Scale a histogram to the expected yield at the configured luminosity."""
    if n_generated <= 0:
        raise ValueError(f"Cannot normalize {hist.GetName()}: N_generated is zero")
    hist.Scale(sigma_pb * LUMI_PB / n_generated)


def draw_overlay(canvas_name, canvas_title, entries, draw_option="HIST E",
                 legend_box=(0.25, 0.70, 0.88, 0.88), text_size=0.030):
    """Draw overlaid histograms.

    entries is a list of (histogram, colour, legend_label, fill) tuples.
    Return the canvas and legend; both are kept alive globally.
    """
    if not entries:
        raise ValueError("At least one histogram is required for an overlay")

    canvas = make_canvas(canvas_name, canvas_title)
    canvas.cd()

    for hist, color, _, fill in entries:
        style_hist(hist, color, fill=fill)

    first_hist = entries[0][0]
    max_value = max(hist.GetMaximum() for hist, _, _, _ in entries)
    first_hist.SetMinimum(0.0)
    first_hist.SetMaximum(max(1.0, max_value * 1.40))
    first_hist.GetYaxis().SetTitleOffset(1.4)

    for index, (hist, _, _, _) in enumerate(entries):
        hist.Draw(draw_option if index == 0 else f"{draw_option} SAME")

    legend = make_legend(*legend_box, text_size=text_size)
    for hist, _, label, option_fill in entries:
        legend.AddEntry(hist, f"{label} [Int: {hist.Integral():.4g}]", "lf" if option_fill else "l")
    legend.Draw()
    draw_experiment_label()
    canvas.Modified()
    canvas.Update()
    canvas.Draw()
    return canvas, legend


def draw_stacked(canvas_name, canvas_title, entries, x_title):
    """Draw a stacked distribution and its total statistical uncertainty band."""
    canvas = make_canvas(canvas_name, canvas_title)
    canvas.cd()

    stack = ROOT.THStack(f"{canvas_name}_stack", f";{x_title};{Y_TITLE}")
    for hist, color, _ in entries:
        style_hist(hist, color, fill=True)
        stack.Add(hist)

    total = entries[0][0].Clone(f"{canvas_name}_total")
    for hist, _, _ in entries[1:]:
        total.Add(hist)
    total.SetFillStyle(3004)
    total.SetFillColor(ROOT.kGray + 2)
    total.SetLineColor(ROOT.kGray + 2)
    total.SetMarkerSize(0)
    ROOT_OBJECTS.extend([stack, total])

    stack.Draw("HIST")
    stack.GetXaxis().SetTitle(x_title)
    stack.GetYaxis().SetTitle(Y_TITLE)
    stack.GetYaxis().SetTitleOffset(1.4)
    stack.SetMinimum(0.0)
    stack.SetMaximum(max(1.0, stack.GetMaximum() * 1.40))
    total.Draw("E2 SAME")

    legend = make_legend(0.25, 0.66, 0.88, 0.88, text_size=0.028)
    for hist, _, label in reversed(entries):
        legend.AddEntry(hist, f"{label} [Int: {hist.Integral():.4g}]", "lf")
    legend.AddEntry(total, f"MC statistical uncertainty [Int: {total.Integral():.4g}]", "f")
    legend.Draw()

    draw_experiment_label()
    canvas.Modified()
    canvas.Update()
    canvas.Draw()
    return canvas, legend, stack, total




## 3. W production-angle analysis

In [ ]:
# ---------------------------------------------------------------------------
# Production-angle analysis
# ---------------------------------------------------------------------------
def make_production_hist(name):
    x_min, x_max = OBSERVABLE_RANGES["cos_theta_w_abs"]
    return make_hist(name, TITLE_COS_THETA_W, x_min, x_max)


def find_w_index(event, pdg_id, event_number):
    """Find the unique W boson with the requested PDG ID in an LHE event."""
    candidates = [
        index for index in range(event.nLHEPart)
        if event.LHEPart_pdgId[index] == pdg_id
    ]

    if len(candidates) > 1 and hasattr(event, "LHEPart_status"):
        final_candidates = [
            index for index in candidates
            if event.LHEPart_status[index] == 1
        ]
        if len(final_candidates) == 1:
            candidates = final_candidates

    if len(candidates) != 1:
        raise RuntimeError(
            f"Event {event_number}: found {len(candidates)} W bosons "
            f"with PDG ID {pdg_id}; expected exactly one."
        )
    return candidates[0]


def particle_vector(event, index):
    """Build a TLorentzVector from one LHE particle."""
    vector = ROOT.TLorentzVector()
    vector.SetPtEtaPhiM(
        event.LHEPart_pt[index],
        event.LHEPart_eta[index],
        event.LHEPart_phi[index],
        event.LHEPart_mass[index],
    )
    return vector


def fill_abs_cos_theta_w(tree, hist):
    """Fill |cos(theta_W)| for both W bosons in every generated event."""
    n_events = 0
    n_bosons = 0

    for event_number, event in enumerate(tree):
        index_plus = find_w_index(event, 24, event_number)
        index_minus = find_w_index(event, -24, event_number)

        for index in (index_plus, index_minus):
            w_vector = particle_vector(event, index)
            momentum = w_vector.P()
            if momentum <= 0.0:
                raise RuntimeError(f"Event {event_number}: W momentum is zero")

            cos_theta = w_vector.Pz() / momentum
            cos_theta = max(-1.0, min(1.0, cos_theta))
            abs_cos_theta = min(abs(cos_theta), 1.0 - 1.0e-12)
            hist.Fill(abs_cos_theta)
            n_bosons += 1

        n_events += 1

    return n_events, n_bosons


def build_production_histograms():
    """Build and luminosity-normalize the 00, TT, and combined mixed samples."""
    components = {
        "00": [("production_00", "00")],
        "TT": [("production_tt", "TT")],
        "mixed": [("production_mixed_1", "0T"), ("production_mixed_2", "T0")],
    }
    histograms = {}

    for component, inputs in components.items():
        combined = make_production_hist(f"h_production_{component}")

        for sample_role, sigma_key in inputs:
            _, tree = open_tree(sample_role)
            n_generated = int(tree.GetEntries())
            if n_generated <= 0:
                raise ValueError(f"Sample {sample_role!r} contains no generated events")

            part = make_production_hist(f"h_production_{component}_{sigma_key}")
            n_events, n_bosons = fill_abs_cos_theta_w(tree, part)
            sigma = PRODUCTION_CROSS_SECTIONS_PB[sigma_key]
            weight = sigma * LUMI_PB / n_generated
            part.Scale(weight)
            combined.Add(part)

            print(
                f"{sample_role}: {n_events} events, {n_bosons} W bosons, "
                f"event weight = {weight:.6e}"
            )

        histograms[component] = combined

    expected = {
        "00": 2.0 * PRODUCTION_CROSS_SECTIONS_PB["00"] * LUMI_PB,
        "TT": 2.0 * PRODUCTION_CROSS_SECTIONS_PB["TT"] * LUMI_PB,
        "mixed": 2.0 * (
            PRODUCTION_CROSS_SECTIONS_PB["0T"]
            + PRODUCTION_CROSS_SECTIONS_PB["T0"]
        ) * LUMI_PB,
    }

    print("\nProduction-angle integral check (including under/overflow):")
    for component, hist in histograms.items():
        observed = hist.Integral(0, hist.GetNbinsX() + 1)
        print(
            f"  {LABELS[component]}: observed = {observed:.2f}, "
            f"expected = {expected[component]:.2f}"
        )

    return histograms


def plot_production_histograms(histograms):
    entries = [
        (histograms["00"], COLORS["00"], LABELS["00"], True),
        (histograms["TT"], COLORS["TT"], LABELS["TT"], True),
        (histograms["mixed"], COLORS["mixed"], LABELS["mixed"], True),
    ]
    return draw_overlay(
        "c_production_angle",
        "Combined W production angle",
        entries,
        draw_option="HIST E",
    )




## 4. W decay angles and dilepton kinematics

In [ ]:
# ---------------------------------------------------------------------------
# Decay-angle and dilepton-kinematics analysis
# ---------------------------------------------------------------------------
def make_decay_hist(name):
    x_min, x_max = OBSERVABLE_RANGES["cos_theta_star"]
    return make_hist(name, TITLE_COS_THETA_STAR, x_min, x_max)


def fill_charge_separated_decay_angles(tree, hist_wplus, hist_wminus):
    """Fill cos(theta*) separately for W+ -> l+ nu and W- -> l- anti-nu.

    The generator-level particle selection and pT threshold follow the original
    analysis: particles with pT below 0.1 GeV are ignored.
    """
    for event in tree:
        wplus_vector = None
        wminus_vector = None
        positive_lepton = None
        negative_lepton = None

        for index in range(event.nLHEPart):
            pdg_id = int(event.LHEPart_pdgId[index])
            pt = event.LHEPart_pt[index]
            if pt < 0.1:
                continue

            vector = particle_vector(event, index)
            if pdg_id == 24:
                wplus_vector = vector
            elif pdg_id == -24:
                wminus_vector = vector
            elif pdg_id in (-11, -13):
                positive_lepton = vector
            elif pdg_id in (11, 13):
                negative_lepton = vector

        if wplus_vector is not None and positive_lepton is not None:
            lepton_rest = ROOT.TLorentzVector(positive_lepton)
            lepton_rest.Boost(-wplus_vector.BoostVector())
            p_rest = lepton_rest.Vect()
            p_w_lab = wplus_vector.Vect()
            if p_rest.Mag() > 0.0 and p_w_lab.Mag() > 0.0:
                value = p_rest.Dot(p_w_lab) / (p_rest.Mag() * p_w_lab.Mag())
                hist_wplus.Fill(max(-1.0, min(1.0, value)))

        if wminus_vector is not None and negative_lepton is not None:
            lepton_rest = ROOT.TLorentzVector(negative_lepton)
            lepton_rest.Boost(-wminus_vector.BoostVector())
            p_rest = lepton_rest.Vect()
            p_w_lab = wminus_vector.Vect()
            if p_rest.Mag() > 0.0 and p_w_lab.Mag() > 0.0:
                value = p_rest.Dot(p_w_lab) / (p_rest.Mag() * p_w_lab.Mag())
                hist_wminus.Fill(max(-1.0, min(1.0, value)))


def get_charged_leptons(event):
    """Return e/mu charged leptons with pT > 0.1 GeV as four-vectors."""
    leptons = []
    pdg_ids = []
    for index in range(event.nLHEPart):
        pdg_id = int(event.LHEPart_pdgId[index])
        if abs(pdg_id) not in (11, 13) or event.LHEPart_pt[index] <= 0.1:
            continue
        leptons.append(particle_vector(event, index))
        pdg_ids.append(pdg_id)
    return leptons, pdg_ids


def fill_dilepton_kinematics(tree, hist_leading_pt, hist_subleading_pt, hist_mll):
    """Fill leading/subleading lepton pT and mll for events with exactly two leptons."""
    for event in tree:
        leptons, _ = get_charged_leptons(event)
        if len(leptons) != 2:
            continue

        pt_values = (leptons[0].Pt(), leptons[1].Pt())
        hist_leading_pt.Fill(max(pt_values))
        hist_subleading_pt.Fill(min(pt_values))
        hist_mll.Fill((leptons[0] + leptons[1]).M())


def build_decay_histograms():
    """Build charge-separated decay and kinematic histograms for each signal sample."""
    sample_keys = {
        "00": ("decay_00", CROSS_SECTIONS_PB["00"]),
        "TT": ("decay_tt", CROSS_SECTIONS_PB["TT"]),
        "0T": ("decay_0t", CROSS_SECTIONS_PB["0T"]),
        "T0": ("decay_t0", CROSS_SECTIONS_PB["0T"]),
    }
    ranges = {
        "lep_pt": OBSERVABLE_RANGES["lep_pt"],
        "subleading_pt": OBSERVABLE_RANGES["subleading_lepton_pt"],
        "mll": OBSERVABLE_RANGES["mll"],
    }
    x_titles = {
        "lep_pt": TITLE_LEADING_PT,
        "subleading_pt": TITLE_SUBLEADING_PT,
        "mll": TITLE_MLL,
    }

    output = {}
    for component, (sample_role, sigma) in sample_keys.items():
        _, tree = open_tree(sample_role)
        n_generated = int(tree.GetEntries())
        if n_generated <= 0:
            raise ValueError(f"Sample {sample_role!r} contains no generated events")

        histograms = {
            "wplus": make_decay_hist(f"h_wplus_{component}"),
            "wminus": make_decay_hist(f"h_wminus_{component}"),
        }
        for variable, (x_min, x_max) in ranges.items():
            histograms[variable] = make_hist(
                f"h_{variable}_{component}",
                x_titles[variable],
                x_min,
                x_max,
            )

        fill_charge_separated_decay_angles(
            tree, histograms["wplus"], histograms["wminus"]
        )
        fill_dilepton_kinematics(
            tree,
            histograms["lep_pt"],
            histograms["subleading_pt"],
            histograms["mll"],
        )

        for hist in histograms.values():
            scale_to_luminosity(hist, sigma, n_generated)

        output[component] = histograms

    # Combine the two mixed polarization samples after separate normalization.
    output["mixed"] = {}
    for variable in ("wplus", "wminus", "lep_pt", "subleading_pt", "mll"):
        combined = output["0T"][variable].Clone(f"h_{variable}_mixed")
        combined.Add(output["T0"][variable])
        combined.SetStats(False)
        ROOT_OBJECTS.append(combined)
        output["mixed"][variable] = combined

    return output


def plot_decay_angle_overlays(decay_histograms):
    results = []
    for charge, key, title in (
        ("W+", "wplus", "W+ decay angle"),
        ("W-", "wminus", "W- decay angle"),
    ):
        entries = [
            (decay_histograms[component][key], COLORS[component], f"{LABELS[component]} ({charge})", True)
            for component in ("00", "TT", "mixed")
        ]
        results.append(
            draw_overlay(
                f"c_decay_{key}",
                title,
                entries,
                draw_option="HIST E",
                legend_box=(0.25, 0.72, 0.88, 0.88),
            )
        )
    return results


def plot_decay_stacks(decay_histograms):
    output = []
    for charge, key, title in (
        ("W+", "wplus", "W+ decay angle - stacked"),
        ("W-", "wminus", "W- decay angle - stacked"),
    ):
        entries = [
            (decay_histograms[component][key], COLORS[component], f"{LABELS[component]} ({charge})")
            for component in ("00", "mixed", "TT")
        ]
        output.append(
            draw_stacked(
                f"c_stack_{key}",
                title,
                entries,
                TITLE_COS_THETA_STAR,
            )
        )
    return output


def plot_kinematic_overlays(decay_histograms):
    output = []
    variables = (
        ("lep_pt", "c_leading_lepton_pt", "Leading lepton transverse momentum", TITLE_LEADING_PT),
        ("subleading_pt", "c_subleading_lepton_pt", "Subleading lepton transverse momentum", TITLE_SUBLEADING_PT),
        ("mll", "c_dilepton_mass", "Dilepton invariant mass", TITLE_MLL),
    )

    for variable, canvas_name, title, _ in variables:
        entries = [
            (decay_histograms[component][variable], COLORS[component], LABELS[component], True)
            for component in ("00", "TT", "mixed")
        ]
        output.append(
            draw_overlay(
                canvas_name,
                title,
                entries,
                draw_option="HIST E",
                legend_box=(0.48, 0.72, 0.88, 0.88),
            )
        )
    return output




## 5. Full-background comparison

In [ ]:
# ---------------------------------------------------------------------------
# Full-background comparison
# ---------------------------------------------------------------------------
LEPTON_NAMES = {
    11: "e-",
    -11: "e+",
    13: "mu-",
    -13: "mu+",
}


def fill_full_background(tree, histograms):
    """Fill the full e/mu background, retaining events with exactly two leptons.

    All dilepton flavour/charge combinations are retained, matching the supplied
    analysis code. The background normalization uses the total generated event
    count, not the number passing this two-lepton selection.
    """
    state_counts = defaultdict(int)
    n_rejected = 0

    for event in tree:
        leptons, pdg_ids = get_charged_leptons(event)
        if len(leptons) != 2:
            n_rejected += 1
            continue

        state = tuple(sorted(pdg_ids))
        state_counts[state] += 1

        dilepton = leptons[0] + leptons[1]
        histograms["mll"].Fill(dilepton.M())
        histograms["lep_pt"].Fill(max(leptons[0].Pt(), leptons[1].Pt()))

        dilepton_momentum = dilepton.Vect()
        beta_dilepton = dilepton.BoostVector()
        if dilepton_momentum.Mag() > 0.0 and beta_dilepton.Mag2() < 1.0:
            for lepton in leptons:
                lepton_rest = ROOT.TLorentzVector(lepton)
                lepton_rest.Boost(-beta_dilepton)
                p_rest = lepton_rest.Vect()
                if p_rest.Mag() > 0.0:
                    value = p_rest.Dot(dilepton_momentum) / (
                        p_rest.Mag() * dilepton_momentum.Mag()
                    )
                    histograms["cos_theta_star"].Fill(max(-1.0, min(1.0, value)))

    return dict(state_counts), n_rejected


def build_background_histograms():
    _, tree = open_tree("background")
    n_generated = int(tree.GetEntries())
    if n_generated <= 0:
        raise ValueError("The configured background sample contains no generated events")

    ranges = {
        "cos_theta_star": OBSERVABLE_RANGES["cos_theta_star"],
        "lep_pt": OBSERVABLE_RANGES["lep_pt"],
        "mll": OBSERVABLE_RANGES["mll"],
    }
    titles = {
        "cos_theta_star": TITLE_COS_THETA_STAR,
        "lep_pt": TITLE_LEADING_PT,
        "mll": TITLE_MLL,
    }
    histograms = {
        variable: make_hist(
            f"h_background_{variable}",
            titles[variable],
            *ranges[variable],
        )
        for variable in ranges
    }

    state_counts, n_rejected = fill_full_background(tree, histograms)
    sigma_background = CROSS_SECTIONS_PB["background"]
    for hist in histograms.values():
        scale_to_luminosity(hist, sigma_background, n_generated)

    print("\nFull-background composition")
    n_selected = sum(state_counts.values())
    print(f"  Generated events: {n_generated}")
    print(f"  Events with exactly two charged leptons: {n_selected}")
    print(f"  Rejected events: {n_rejected}")
    print(f"\n{'Final state':<18}{'Events':>10}{'Fraction':>12}{'Sigma [pb]':>14}")

    for state, count in sorted(state_counts.items(), key=lambda item: -item[1]):
        state_name = " ".join(LEPTON_NAMES.get(pdg_id, str(pdg_id)) for pdg_id in state)
        fraction = count / n_generated if n_generated > 0 else 0.0
        sigma_state = sigma_background * fraction
        print(f"{state_name:<18}{count:>10d}{fraction:>12.3f}{sigma_state:>14.5f}")
    print()

    return histograms


def make_signal_comparison_histograms(decay_histograms):
    """Combine charge-separated decay angles and expose signal templates by variable."""
    samples = {}
    for component in ("00", "TT", "0T", "T0"):
        cos_hist = decay_histograms[component]["wplus"].Clone(f"h_cos_theta_star_{component}")
        cos_hist.Add(decay_histograms[component]["wminus"])
        cos_hist.SetStats(False)
        ROOT_OBJECTS.append(cos_hist)

        samples[component] = {
            "cos_theta_star": cos_hist,
            "lep_pt": decay_histograms[component]["lep_pt"],
            "mll": decay_histograms[component]["mll"],
        }

    samples["mixed"] = {}
    for variable in ("cos_theta_star", "lep_pt", "mll"):
        mixed = samples["0T"][variable].Clone(f"h_{variable}_mixed_for_comparison")
        mixed.Add(samples["T0"][variable])
        mixed.SetStats(False)
        ROOT_OBJECTS.append(mixed)
        samples["mixed"][variable] = mixed

    return samples


def plot_signal_vs_background(signal_histograms, background_histograms):
    output = []
    specifications = (
        ("cos_theta_star", "c_components_vs_background_cos", "Decay angle: signal components vs background"),
        ("lep_pt", "c_components_vs_background_pt", "Leading lepton pT: signal components vs background"),
        ("mll", "c_components_vs_background_mll", "Dilepton mass: signal components vs background"),
    )

    for variable, canvas_name, title in specifications:
        entries = [
            (signal_histograms[component][variable], COLORS[component], LABELS[component], True)
            for component in ("00", "TT", "mixed")
        ]
        entries.append(
            (background_histograms[variable], COLORS["bkg"], LABELS["bkg"], False)
        )
        result = draw_overlay(
            canvas_name,
            title,
            entries,
            draw_option="HIST E",
            legend_box=(0.38, 0.66, 0.89, 0.87),
            text_size=0.026,
        )
        output.append(result)

        signal_yield = sum(
            signal_histograms[component][variable].Integral()
            for component in ("00", "TT", "mixed")
        )
        background_yield = background_histograms[variable].Integral()
        ratio = background_yield / signal_yield if signal_yield > 0 else float("nan")
        print(
            f"{variable}: signal = {signal_yield:.4g}, "
            f"background = {background_yield:.4g}, "
            f"background/signal = {ratio:.4g}"
        )

    return output




## 6. Save templates and plots; run the analysis

In [ ]:
# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------
def save_templates(signal_histograms, background_histograms):
    """Save the three signal components and the background templates to ROOT."""
    output_file = ROOT.TFile.Open(TEMPLATE_FILE, "RECREATE")
    if not output_file or output_file.IsZombie():
        raise OSError(f"Could not create ROOT template file: {TEMPLATE_FILE}")

    output_file.cd()
    for variable in ("cos_theta_star", "lep_pt", "mll"):
        output_names = {
            "00": "00",
            "TT": "tt",
            "mixed": "mixed",
        }
        for component, output_name in output_names.items():
            signal_histograms[component][variable].Write(
                f"h_{variable}_{output_name}"
            )
        background_histograms[variable].Write(f"h_{variable}_bkg")

    output_file.Close()
    print(f"\nROOT templates saved to: {TEMPLATE_FILE}")


def save_canvases():
    """Save all ROOT canvases as both PDF and PNG in the configured output folder."""
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    canvases = ROOT.gROOT.GetListOfCanvases()
    for canvas in canvases:
        for extension in ("pdf", "png"):
            destination = PLOTS_DIR / f"{canvas.GetName()}.{extension}"
            canvas.SaveAs(str(destination))
    print(f"Plots saved to: {PLOTS_DIR}")


def main():
    """Run the full angular, kinematic, and signal/background analysis."""
    print("Starting angular and kinematic analysis")
    print(f"Integrated luminosity: {LUMI_PB:g} pb^-1")
    print(f"Input directory: {Path(sample_path('decay_00')).parent}")

    production_histograms = build_production_histograms()
    plot_production_histograms(production_histograms)

    decay_histograms = build_decay_histograms()
    plot_decay_angle_overlays(decay_histograms)
    plot_decay_stacks(decay_histograms)
    plot_kinematic_overlays(decay_histograms)

    background_histograms = build_background_histograms()
    signal_histograms = make_signal_comparison_histograms(decay_histograms)
    plot_signal_vs_background(signal_histograms, background_histograms)

    save_templates(signal_histograms, background_histograms)
    save_canvases()

    print("Analysis completed successfully.")


if __name__ == "__main__":
    main()
